# Reducción de dimensionalidad sobre MNIST

Este notebook carga el dataset **MNIST** y aplica varias técnicas de reducción de dimensionalidad para visualizar la estructura de los datos:

- **PCA**
- **t-SNE**
- **UMAP**

Además, incluye visualizaciones para comparar los métodos y entender qué patrones capturan.


## 1. Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml
import umap
from sklearn.model_selection import train_test_split


plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True
np.random.seed(42)


## 2. Carga de datos

MNIST contiene imágenes de dígitos escritos a mano de tamaño **28x28**, que convertiremos en vectores de 784 dimensiones.


In [ ]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

X = mnist.data.astype(np.float32) / 255.0
y = mnist.target.astype(np.int64)

print('Dataset completo:', X.shape)

# split train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=10000,
    random_state=42,
    stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

# reconstruir imágenes
X_train_images = X_train.reshape(-1, 28, 28)

print('Datos aplanados:', X_train.shape)

## 3. Visualización de ejemplos


In [ ]:
X_train_images = X_train.reshape(-1, 28, 28)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_train_images[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')

plt.suptitle('Ejemplos del dataset MNIST', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Submuestreo para visualización

t-SNE y UMAP pueden tardar bastante con decenas de miles de muestras. Vamos a trabajar con un subconjunto para que el notebook sea ágil y visual.


In [ ]:
n_samples = 3000
idx = np.random.choice(len(X), n_samples, replace=False)
X_sub = X[idx]
y_sub = y[idx]

print('Submuestra:', X_sub.shape)


## 5. PCA

PCA busca las direcciones lineales de máxima varianza. Primero veremos cuánta varianza explican las componentes principales, y luego proyectaremos a 2D.


In [ ]:
pca_full = PCA().fit(X_sub)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(9, 5))
plt.plot(cum_var)
plt.xlabel('Número de componentes')
plt.ylabel('Varianza explicada acumulada')
plt.title('PCA - Varianza explicada acumulada')
plt.show()

for threshold in [0.80, 0.90, 0.95]:
    n_comp = np.argmax(cum_var >= threshold) + 1
    print(f'Componentes para {int(threshold*100)}% de varianza: {n_comp}')


In [ ]:
markers = ['o','s','^','v','D','P','X','*','<','>']
colors = plt.cm.tab10(range(10))

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from matplotlib.offsetbox import AnnotationBbox, OffsetImage

def plot_digits(X, y, min_distance=0.04, images=None, figsize=(13, 10)):
    # Let's scale the input features so that they range from 0 to 1
    X_normalized = MinMaxScaler().fit_transform(X)
    # Now we create the list of coordinates of the digits plotted so far.
    # We pretend that one is already plotted far away at the start, to
    # avoid `if` statements in the loop below
    neighbors = np.array([[10., 10.]])
    # The rest should be self-explanatory
    plt.figure(figsize=figsize)
    cmap = plt.cm.jet
    digits = np.unique(y)
    for digit in digits:
        plt.scatter(X_normalized[y == digit, 0], X_normalized[y == digit, 1],
                    c=[cmap(float(digit) / 9)], alpha=0.5)
    plt.axis("off")
    ax = plt.gca()  # get current axes
    for index, image_coord in enumerate(X_normalized):
        closest_distance = np.linalg.norm(neighbors - image_coord, axis=1).min()
        if closest_distance > min_distance:
            neighbors = np.r_[neighbors, [image_coord]]
            if images is None:
                plt.text(image_coord[0], image_coord[1], str(int(y[index])),
                         color=cmap(float(y[index]) / 9),
                         fontdict={"weight": "bold", "size": 12})
            else:
                image = images[index].reshape(28, 28)
                imagebox = AnnotationBbox(OffsetImage(image, cmap="binary"),
                                          image_coord)
                ax.add_artist(imagebox)

In [ ]:
images_sub = X_sub.reshape(-1, 28, 28)

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_sub)

plot_digits(X_pca_2d, y_sub, min_distance=0.04, images=None, figsize=(13, 10))
plt.title("MNIST con PCA", fontsize=16)
plt.show()

In [ ]:
plot_digits(X_pca_2d, y_sub, min_distance=0.04, images=images_sub, figsize=(35,25))
plt.title("MNIST con PCA (con imágenes)", fontsize=16)
plt.show()

In [ ]:
# Número de imágenes a mostrar
n_samples = 10

# Seleccionamos una pequeña muestra
X_sample = X_sub[:n_samples]

# PCA con distintos números de componentes
pca_10 = PCA(n_components=6, random_state=42)
pca_40 = PCA(n_components=40, random_state=42)
pca_100 = PCA(n_components=100, random_state=42)

# Ajustar y reconstruir
X_10 = pca_10.fit_transform(X_sub)
X_recon_10 = pca_10.inverse_transform(pca_10.transform(X_sample))

X_40 = pca_40.fit_transform(X_sub)
X_recon_40 = pca_40.inverse_transform(pca_40.transform(X_sample))

X_100 = pca_100.fit_transform(X_sub)
X_recon_100 = pca_100.inverse_transform(pca_100.transform(X_sample))

# Plot
fig, axes = plt.subplots(4, n_samples, figsize=(15, 6))

for i in range(n_samples):

    # Original
    axes[0, i].imshow(X_sample[i].reshape(28,28), cmap="gray")
    axes[0, i].axis("off")

    # PCA 10
    axes[1, i].imshow(X_recon_10[i].reshape(28,28), cmap="gray")
    axes[1, i].axis("off")

    # PCA 40
    axes[2, i].imshow(X_recon_40[i].reshape(28,28), cmap="gray")
    axes[2, i].axis("off")

    # PCA 100
    axes[3, i].imshow(X_recon_100[i].reshape(28,28), cmap="gray")
    axes[3, i].axis("off")

axes[0,0].set_ylabel("Original", fontsize=12)
axes[1,0].set_ylabel("6 comp", fontsize=12)
axes[2,0].set_ylabel("40 comp", fontsize=12)
axes[3,0].set_ylabel("100 comp", fontsize=12)

plt.suptitle("Reconstrucción de MNIST con PCA", fontsize=16)
plt.tight_layout()
plt.show()

## 6. t-SNE

t-SNE es una técnica no lineal muy potente para visualización local. Suele separar bien los clústeres, aunque no preserva tan bien la estructura global.

Como buena práctica, aplicamos primero PCA a 50 dimensiones para acelerar el proceso.


In [ ]:
pca_50 = PCA(n_components=50, random_state=42)
X_pca_50 = pca_50.fit_transform(X_sub)

tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=42
)

X_tsne = tsne.fit_transform(X_pca_50)

plot_digits(X_tsne, y_sub, min_distance=0.04, images=None, figsize=(13, 10))
plt.title("MNIST con t-SNE", fontsize=16)
plt.show()

In [ ]:
plot_digits(X_tsne, y_sub, min_distance=0.04, images=images_sub, figsize=(35,25))
plt.title("MNIST con t-SNE (con imágenes)", fontsize=16)
plt.show()

In [ ]:
# Filtrar solo dígitos 4 y 9
mask = (y_sub == 4) | (y_sub == 9)

X_49 = X_sub[mask]
y_49 = y_sub[mask]

images_49 = images_sub[mask]

print("Shape:", X_49.shape)
print("Labels únicas:", np.unique(y_49))

In [ ]:
# Reducimos primero con PCA para acelerar t-SNE
pca_30 = PCA(n_components=30, random_state=42)
X_49_pca = pca_30.fit_transform(X_49)

tsne_49 = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=42
)

X_49_tsne = tsne_49.fit_transform(X_49_pca)

plot_digits(X_49_tsne, y_49, min_distance=0.04, images=images_49, figsize=(35,25))
plt.title("t-SNE sobre dígitos 4 y 9", fontsize=16)
plt.show()

## 7. UMAP

UMAP también es una técnica no lineal, pero suele ser más rápida que t-SNE y a menudo preserva mejor cierta estructura global.


In [ ]:
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42
)

X_umap = umap_model.fit_transform(X_sub)

plot_digits(X_umap, y_sub, min_distance=0.04, images=None, figsize=(13, 10))
plt.title("MNIST con UMAP", fontsize=16)
plt.show()

In [ ]:
plot_digits(X_umap, y_sub, min_distance=0.04, images=images_sub, figsize=(35,25))
plt.title("MNIST con UMAP (con imágenes)", fontsize=16)
plt.show()

## 8. Comparación visual

Vamos a comparar los embeddings lado a lado para apreciar diferencias entre métodos:

- **PCA**: rápido, lineal, menos separación
- **t-SNE**: excelente separación local
- **UMAP**: muy visual y normalmente más escalable


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_sub, s=8, alpha=0.7)
axes[0].set_title('PCA')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sub, s=8, alpha=0.7)
axes[1].set_title('t-SNE')
axes[1].set_xlabel('Dim 1')
axes[1].set_ylabel('Dim 2')

axes[2].scatter(X_umap[:, 0], X_umap[:, 1], c=y_sub, s=8, alpha=0.7)
axes[2].set_title('UMAP')
axes[2].set_xlabel('Dim 1')
axes[2].set_ylabel('Dim 2')

plt.suptitle('Comparativa de técnicas de reducción de dimensionalidad en MNIST', fontsize=14)
plt.tight_layout()
plt.show()
